# M5.6 · Autoencoder ailesi — yığılmış, bağlı ağırlıklı, gürültü giderici, seyrek, konvolüsyonel

Kısıtlar altında özdeşlik fonksiyonunu öğrenmenin yan ürünü olarak temsiller: eksik (undercomplete) ve aşırı tam (overcomplete) kodlar, yığılmış (derin) autoencoder'lar, kodlayıcı–kod çözücü ağırlıklarını bağlamak ($W_{N-L+1}=W_L^\top$), açgözlü katman katman ön eğitim, autoencoder ile denetimsiz ön eğitim ve az etiketle sınıflandırma, gürültü giderici (Gauss ve maskeleme gürültüsü), seyrek (L1 ve KL cezası) ve konvolüsyonel/rekürent autoencoder'lar. NumPy'da bağlı ağırlıklı seyrek autoencoder'ın gradyanları elle, PyTorch'ta yığılmış AE, bağlı ağırlık, DAE ve ön eğitimle sınıflandırma, sonda konvolüsyonel vs yoğun AE. Ekler: doğrusal DAE'nin kapalı form çözümü, büzüşen (contractive) AE, yeniden yapılandırma hatasıyla anomali tespiti.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-6-autoencoders.html](https://m-gonul.github.io/pytorch-lab/m5-6-autoencoders.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_6_autoencoders.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        ensure(need(ns, \"ae_params\")([784, 100, 30]) == 163814 and ns[\"ae_params\"]([784, 100, 30], True) == 82414, \"Géron örneği\")\n        close(need(ns, \"kl_sparsity\")(0.05, 0.2), 0.05 * math.log(0.25) + 0.95 * math.log(0.95 / 0.8), what=\"KL\")\n        close(ns[\"kl_sparsity\"](0.05, 0.05), 0.0, what=\"hedefte 0\")\n        h = 1e-7\n        close(need(ns, \"kl_sparsity_grad\")(0.05, 0.2), (ns[\"kl_sparsity\"](0.05, 0.2 + h) - ns[\"kl_sparsity\"](0.05, 0.2)) / h, atol=1e-4, what=\"türev\")\n        ensure(need(ns, \"greedy_plan\")([784, 100, 30]) == [(784, 100), (100, 30)], \"aşamalar\")\n        rng = random.Random(0)\n        x = [1.0] * 10000\n        m = need(ns, \"masking_noise\")(x, 0.5, rng)\n        ensure(abs(sum(v == 0 for v in m) / 10000 - 0.5) < 0.02 and set(m) <= {0.0, 1.0}, \"maskeleme\")\n        gn = need(ns, \"gaussian_noise\")(x, 0.3, rng)\n        mu = sum(gn) / len(gn)\n        close(math.sqrt(sum((v - mu) ** 2 for v in gn) / len(gn)), 0.3, atol=0.01, what=\"Gauss gürültüsü std\")\n        rows = []\n        for rh in (0.01, 0.05, 0.1, 0.2, 0.5, 0.9):\n            rows.append([rh, f\"{ns['kl_sparsity'](0.05, rh):.3f}\", f\"{ns['kl_sparsity_grad'](0.05, rh):+.2f}\", f\"{abs(rh - 0.05):.2f}\"])\n        R_.table(\"Seyreklik cezası, hedef ρ = 0.05\", [\"ρ̂\", \"KL(ρ‖ρ̂)\", \"∂/∂ρ̂\", \"|ρ̂ − ρ| (L1 benzeri)\"], rows,\n                 note=\"KL cezası asimetriktir: ρ̂ 1'e yaklaşırken çok hızlı büyür, 0'a giderken de büyür — nöron tamamen ölmez. Hedefin altında (ρ̂ = 0.01) gradyan negatiftir: aktivasyonu yükseltmeye iter.\")\n    R_.test(\"sayımlar ve cezalar\", t)\n",
 "c": "import numpy as np\nfrom sklearn.datasets import load_digits\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüler\", lambda: (max_loops(src, 0, func=\"tied_sparse_ae\"), max_loops(src, 1, func=\"train\")))\n\n    def t_grad():\n        rng = np.random.default_rng(0)\n        X = rng.random((7, 5))\n        W, b, c = rng.normal(0, 0.5, (5, 3)), rng.normal(size=3), rng.normal(size=5)\n        L, dW, db, dc = need(ns, \"tied_sparse_ae\")(X, W, b, c, 0.1, 0.7)\n        h = 1e-6\n        for P, G in ((W, dW), (b, db), (c, dc)):\n            for idx in np.ndindex(P.shape):\n                P[idx] += h\n                Lp = ns[\"tied_sparse_ae\"](X, W, b, c, 0.1, 0.7)[0]\n                P[idx] -= h\n                close((Lp - L) / h, G[idx], atol=2e-4, what=f\"sayısal gradyan {idx}\")\n    R_.test(\"bağlı ağırlık + KL gradyanı\", t_grad)\n\n    def t_train():\n        X = load_digits().data[:600] / 16\n        rows = []\n        res = {}\n        for beta in (0.0, 0.5, 3.0):\n            W, b, c = need(ns, \"train\")(X, 25, 0.05, beta, 0.5, 1500, 0)\n            z = 1 / (1 + np.exp(-(X @ W + b)))\n            mse = np.mean((z @ W.T + c - X) ** 2)\n            res[beta] = (z.mean(), mse)\n            rows.append([beta, f\"{z.mean():.3f}\", f\"{(z > 0.5).mean():.3f}\", f\"{mse:.4f}\"])\n        ensure(res[3.0][0] < 0.12 and res[0.0][0] > res[3.0][0] + 0.1, \"β ile ortalama aktivasyon hedefe iner\")\n        ensure(res[0.0][1] < np.var(X, 0).mean(), \"AE ortalama tahminden iyi\")\n        R_.table(\"Rakamlar (600 × 64), 25 sigmoid kod nöronu, ρ = 0.05\", [\"β\", \"ortalama aktivasyon\", \"> 0.5 oranı\", \"yeniden yapılandırma MSE\"], rows,\n                 note=\"β = 0'da kod nöronları yarı yarıya açık (yoğun kod); β büyüdükçe her örnekte az sayıda nöron açılır ve ortalama aktivasyon ρ'ya iner. Bedel biraz daha yüksek yeniden yapılandırma hatası — karşılığında her nöron daha seçici (yerel kalem darbesi benzeri) bir özniteliğe karşılık gelir.\")\n    R_.test(\"seyrek eğitim\", t_train)\n",
 "d": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    X, y = need(ns, \"digits\")()\n    g = torch.Generator().manual_seed(0)\n    p = torch.randperm(len(X), generator=g)\n    te, pool = p[:497], p[497:]\n    lab = pool[:50]\n\n    def t():\n        t00 = time.perf_counter()\n        rows, M = [], {}\n        for name, kw, mask in ((\"bağsız AE\", dict(tied=False), 0.0), (\"bağlı AE\", dict(tied=True), 0.0), (\"gürültü giderici AE (maske 0.5)\", dict(tied=False), 0.5)):\n            torch.manual_seed(0)\n            m = need(ns, \"AE\")((64, 32, 12), **kw)\n            need(ns, \"train_ae\")(m, X[pool], 100, 3e-3, mask)\n            M[name] = m\n        torch.manual_seed(0)\n        M[\"açgözlü katman katman\"] = need(ns, \"greedy_pretrain\")((64, 32, 12), X[pool], 100)\n        mk = (torch.rand(X[te].shape, generator=torch.Generator().manual_seed(5)) >= 0.5).float()\n        with torch.no_grad():\n            for name, m in M.items():\n                rows.append([name, sum(q.numel() for q in m.parameters()), f\"{F.mse_loss(m(X[te]), X[te]).item():.4f}\", f\"{F.mse_loss(m(X[te] * mk), X[te]).item():.4f}\"])\n        cnt = {r[0]: r[1] for r in rows}\n        ensure(cnt[\"bağlı AE\"] < 0.6 * cnt[\"bağsız AE\"], \"bağlı ağırlık ≈ yarı parametre\")\n        ensure(float(rows[2][3]) < 0.7 * float(rows[0][3]), \"DAE maskeli girdiyi daha iyi tamamlar\")\n        ensure(float(rows[1][2]) < 1.3 * float(rows[0][2]), \"bağlı AE benzer hata\")\n        R_.table(\"Rakamlar 64 → 32 → 12 → 32 → 64 (1300 eğitim, 100 epoch)\", [\"model\", \"parametre\", \"test MSE\", \"yarısı silinmiş girdiden MSE\"], rows,\n                 note=\"Bağlı ağırlıklar parametreyi yarıya indirir, hata neredeyse aynı kalır. Gürültü giderici AE temiz girdide biraz kötü olabilir ama pikselleri yarısı silinmiş girdiden rakamı yeniden kurar: eksik ayrıntıyı tahmin etmeyi öğrenmiştir.\")\n        acc = {}\n        rows2 = []\n        for name, enc, fr in ((\"sıfırdan\", None, False), (\"AE ön eğitimi + ince ayar\", M[\"bağsız AE\"], False), (\"AE kodlayıcı donuk\", M[\"bağsız AE\"], True),\n                              (\"DAE kodlayıcı donuk\", M[\"gürültü giderici AE (maske 0.5)\"], True), (\"DAE ön eğitimi + ince ayar\", M[\"gürültü giderici AE (maske 0.5)\"], False)):\n            acc[name] = need(ns, \"classify\")(enc, X[lab], y[lab], X[te], y[te], fr, 300)\n            rows2.append([name, f\"{acc[name]:.3f}\"])\n        ensure(acc[\"AE ön eğitimi + ince ayar\"] > acc[\"sıfırdan\"] - 0.01 and acc[\"DAE kodlayıcı donuk\"] > acc[\"AE kodlayıcı donuk\"] - 0.02 and acc[\"AE kodlayıcı donuk\"] > 0.4, f\"ön eğitim {acc}\")  # farklı donanımda ±0.02 oynar\n        R_.table(f\"50 etiketle sınıflandırma ({time.perf_counter() - t00:.0f} sn toplam)\", [\"kodlayıcı\", \"test doğruluğu\"], rows2,\n                 note=\"Etiketsiz 1300 örnekten öğrenilen kodlayıcı az etiketle başlangıç noktası olarak kullanılabilir; bu küçük problemde kazanç küçüktür ve donanım/tohuma göre birkaç puan oynar (tabloyu tek koşu olarak değil, eğilim olarak oku). Donuk kodlarda DAE'nin temsili genelde düz AE'ninkinden iyidir: gürültü giderme kriteri daha 'anlamlı' öznitelikler öğretir (Vincent ve ark. 2010). Güvenilir bir karşılaştırma için birkaç tohumla ortalama al (M8.2 x3).\")\n    R_.test(\"AE ailesi ve ön eğitim\", t)\n",
 "e": "import time\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.datasets import load_digits\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        X = torch.tensor(load_digits().data, dtype=torch.float32).view(-1, 1, 8, 8) / 16\n        X = F.interpolate(X, size=16, mode=\"bilinear\", align_corners=False)\n        g = torch.Generator().manual_seed(0)\n        p = torch.randperm(len(X), generator=g)\n        tr, te = X[p[497:]], X[p[:497]]\n        base = F.mse_loss(tr.mean(0, keepdim=True).expand_as(te), te).item()\n        rows = []\n        res = {}\n        for name in (\"ConvAE\", \"DenseAE\"):\n            torch.manual_seed(0)\n            m = need(ns, name)(16)\n            ensure(tuple(m(te[:2]).shape) == (2, 1, 16, 16), f\"{name} çıkış şekli\")\n            opt = torch.optim.Adam(m.parameters(), 3e-3)\n            t0 = time.perf_counter()\n            for _ in range(60):\n                perm = torch.randperm(len(tr))\n                for i in range(0, len(tr), 64):\n                    xb = tr[perm[i:i + 64]]\n                    loss = F.mse_loss(m(xb), xb)\n                    opt.zero_grad()\n                    loss.backward()\n                    opt.step()\n            with torch.no_grad():\n                mse = F.mse_loss(m(te), te).item()\n            res[name] = (sum(q.numel() for q in m.parameters()), mse)\n            rows.append([name, res[name][0], f\"{mse:.4f}\", f\"{time.perf_counter() - t0:.1f} sn\"])\n        ensure(res[\"ConvAE\"][1] < base / 3 and res[\"DenseAE\"][1] < base / 3, \"ikisi de ortalama görüntüden çok iyi\")\n        ensure(res[\"ConvAE\"][0] < res[\"DenseAE\"][0], \"konvolüsyonel daha az parametre\")\n        R_.table(f\"16 × 16 rakamlar, 16 boyutlu kod, 60 epoch (ortalama görüntü MSE = {base:.4f})\", [\"model\", \"parametre\", \"test MSE\", \"süre\"], rows,\n                 note=\"Konvolüsyonel kodlayıcı uzamsal boyutu azaltırken derinliği artırır, kod çözücü transposed conv ile tersini yapar. Parametrelerin çoğu her iki modelde de kod ile düzleştirilmiş harita arasındaki yoğun katmandadır. Büyük görüntülerde yoğun AE'nin parametresi piksel sayısıyla patlar; konvolüsyonel olan ölçeklenir.\")\n    R_.test(\"konvolüsyonel vs yoğun\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        d = 5\n        Q, _ = np.linalg.qr(rng.normal(size=(d, d)))\n        lam = np.array([4.0, 2.0, 1.0, 0.3, 0.05])\n        X = rng.normal(size=(20000, d)) * np.sqrt(lam) @ Q.T\n        S = X.T @ X / len(X)\n        Wst = need(ns, \"dae_optimal\")(S, 0.5)\n        W = need(ns, \"train_linear_dae\")(X, 0.5, 0.1, 1500, 0)\n        close(W, Wst, atol=0.03, what=\"SGD ile eğitilen W ≈ kapalı form\")\n        close(ns[\"dae_optimal\"](S, 0.0), np.eye(d), atol=1e-9, what=\"gürültüsüz aşırı tam AE: özdeşlik\")\n        ev = np.linalg.eigvalsh(Wst)[::-1]\n        le = np.linalg.eigvalsh(S)[::-1]\n        rows = [[f\"{l:.2f}\", f\"{e:.3f}\", f\"{l / (l + 0.5):.3f}\"] for l, e in zip(le, ev)]\n        close(ev, le / (le + 0.5), atol=1e-6, what=\"özdeğer büzülmesi λ/(λ + σ²)\")\n        R_.table(\"Doğrusal DAE (σ² = 0.5): W*'ın özdeğerleri\", [\"veri özdeğeri λ\", \"W* özdeğeri\", \"λ/(λ + σ²)\"], rows,\n                 note=\"Kod girdiyle aynı boyutta (aşırı tam) ve gürültüsüz bir doğrusal AE özdeşliği öğrenir — işe yaramaz. Gürültü eklenince çözüm, verinin her ana yönünü λ/(λ + σ²) kadar büzen bir Wiener filtresidir: yüksek varyanslı yönler korunur, gürültü düzeyinin altındakiler bastırılır — PCA'nın 'yumuşak' sürümü. Ridge regresyonla aynı matematik.\")\n    R_.test(\"doğrusal DAE\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"encoder_jacobian\", \"cae_penalty\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        W, b = rng.normal(size=(4, 6)), rng.normal(size=4)\n        x = rng.normal(size=6)\n        J = need(ns, \"encoder_jacobian\")(x, W, b)\n        f = lambda v: 1 / (1 + np.exp(-(W @ v + b)))\n        Jn = np.stack([(f(x + 1e-6 * e) - f(x)) / 1e-6 for e in np.eye(6)], 1)\n        close(J, Jn, atol=1e-5, what=\"Jacobian\")\n        X = rng.normal(size=(30, 6))\n        close(need(ns, \"cae_penalty\")(X, W, b), np.mean([np.sum(ns[\"encoder_jacobian\"](xx, W, b) ** 2) for xx in X]), what=\"‖J‖²_F ortalaması\")\n        rows = []\n        for s in (0.1, 1.0, 3.0, 10.0):\n            rows.append([s, f\"{ns['cae_penalty'](X, s * W, b):.4f}\"])\n        R_.table(\"Ağırlık ölçeği ve büzülme cezası\", [\"W ölçeği\", \"ort. ‖J‖²_F\"], rows,\n                 note=\"Ceza iki yolla küçülür: küçük ağırlıklar ya da doymuş nöronlar (h ≈ 0 veya 1 → h(1 − h) ≈ 0). Büzüşen AE (Rifai ve ark.) kodun girdideki küçük değişimlere duyarsız olmasını ister ama yeniden yapılandırma bunu her yönde yapmasını engeller: sonuç, veri manifoldu boyunca duyarlı, ona dik yönlerde duyarsız bir kod — DAE ile yakın akrabadır (küçük gürültü limitinde eşdeğer).\")\n    R_.test(\"büzüşen AE\", t)\n",
 "x3": "import numpy as np\nfrom sklearn.datasets import load_digits\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"fit_pca\", \"recon_error\", \"roc_auc\")])\n\n    def t():\n        close(need(ns, \"roc_auc\")([0.1, 0.4, 0.35, 0.8], [0, 0, 1, 1]), 0.75, what=\"sklearn örneği\")\n        close(ns[\"roc_auc\"]([1, 1, 1, 1], [0, 1, 0, 1]), 0.5, what=\"eşit skorlar\")\n        X, y = load_digits(return_X_y=True)\n        X = X / 16\n        normal = y <= 4\n        rng = np.random.default_rng(0)\n        idx = rng.permutation(np.flatnonzero(normal))\n        tr = idx[:600]\n        test = np.concatenate([idx[600:], np.flatnonzero(~normal)])\n        rows = []\n        aucs = {}\n        for k in (1, 3, 8, 20, 40, 64):\n            m, C = need(ns, \"fit_pca\")(X[tr], k)\n            e = need(ns, \"recon_error\")(X[test], m, C)\n            aucs[k] = ns[\"roc_auc\"](e, ~normal[test])\n            rows.append([k, f\"{aucs[k]:.3f}\"])\n        ensure(aucs[20] > 0.85 and aucs[1] < aucs[20] and aucs[40] < aucs[20], f\"AUC {aucs}\")\n        R_.table(\"0–4 rakamlarıyla eğitilen doğrusal AE (PCA), 5–9 'anomali'\", [\"kod boyutu k\", \"ROC AUC\"], rows,\n                 note=\"Yalnız normal veriyle eğitilmiş AE onları iyi yeniden kurar, görmediği yapıdaki girdileri kötü: hata anomali skorudur. Kod çok küçükse normal örnekler de kötü kurulur, kod büyüdükçe anomaliler de iyi kurulmaya başlar; k = d'de (darboğaz yok) her şey mükemmel kurulur, 'hata' yalnız kayan nokta gürültüsüdür ve AUC anlamsızlaşır — darboğaz, tespiti mümkün kılan kısıttır.\")\n    R_.test(\"anomali tespiti\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m5_6_autoencoders as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Verimli temsiller:** 40, 27, 25, 36, … mı yoksa 50, 48, 46, …, 14 mü daha kolay ezberlenir? İkincisi daha uzun ama bir desen var. Satranç ustaları gerçek oyunlardan dizilimleri beş saniyede ezberler, rastgele dizilimleri ezberleyemez (Chase & Simon). Uzun dizileri ezberlemek zor olduğu için desen tanımak işe yarar — autoencoder'ı kısıtlamak onu desen bulmaya iter.

**Kodlayıcı + kod çözücü:** girdi → gizli temsil (kod) → yeniden yapılandırma; çıkış nöron sayısı = girdi. Kayıp: yeniden yapılandırma hatası. Kod girdiden küçükse **eksik (undercomplete)** — en önemli öznitelikleri öğrenmek zorundadır. Doğrusal AE + MSE = PCA alt uzayı (M5.1). **Yığılmış AE** birden çok gizli katman içerir, genellikle kod katmanına göre simetriktir (ör. 784-100-30-100-784); çok güçlü yapılırsa her girdiyi keyfi bir sayıya eşleyip mükemmel ama işe yaramaz bir yeniden yapılandırma öğrenir.

**Ağırlık bağlama:** N katmanlı simetrik AE'de $W_{N-L+1}=W_L^\top$ ($L=1..N/2$): ağırlıklar yarıya iner, eğitim hızlanır, aşırı uyum riski azalır; yeniden yapılandırma bağsızla hemen hemen aynıdır. **Açgözlü katman katman eğitim:** önce sığ bir AE girdiyi öğrenir; tüm veri onunla kodlanır, ikinci AE bu kodları öğrenir; sonra gizli katmanlar sırayla, çıkış katmanları ters sırayla yığılır (eskiden derin ağları eğitmenin tek yoluydu). **Denetimsiz ön eğitim:** çok etiketsiz, az etiketli veride önce yığılmış AE tüm veriyle eğitilir, alt katmanları sınıflandırıcıda yeniden kullanılır (çok az etiket varsa dondurulur).

**Konvolüsyonel AE:** kodlayıcı normal CNN (uzamsal boyut ↓, derinlik ↑), kod çözücü tersi (transposed conv ya da yukarı örnekleme + konv.). **Rekürent AE:** dizi→vektör kodlayıcı, vektör→dizi kod çözücü.

**Aşırı tam AE ve başka kısıtlar:** **gürültü giderici** (Vincent ve ark. 2008/2010): girdiye Gauss gürültüsü ya da dropout benzeri maskeleme eklenir, temiz girdi geri istenir — AE girdide olmayan ayrıntıları tahmin etmeyi öğrenir. **Seyrek AE:** kod katmanında az sayıda aktif nöron (ör. ortalama %5): sigmoid kod + L1 aktivasyon cezası ya da her nöronun ortalama aktivasyonu $\hat\rho_j$ ile hedef $\rho$ arasındaki KL cezası $\beta\sum_j\mathrm{KL}(\rho\Vert\hat\rho_j)$. Her kod nöronu yararlı bir öznitelik temsil etmeye zorlanır ("ayda yalnız birkaç kelime konuşabilseydin, söylemeye değer olanları seçerdin").

## 1 · Elle

<!-- çözüm -->

Bağsız: $(784\cdot100+100)+(100\cdot30+30)+(30\cdot100+100)+(100\cdot784+784)=78\,500+3\,030+3\,100+79\,184=163\,814$. Bağlı: kod çözücü ağırlıkları kaybolur, bias'ları kalır: $78\,500+3\,030+100+784=82\,414$ (≈ yarı).

KL: $0.05\ln0.25+0.95\ln(0.95/0.8)=-0.0693+0.1633=0.0939$. Türev: $-\rho/\hat\rho+(1-\rho)/(1-\hat\rho)=-0.25+1.1875=0.9375>0$ → gradyan inişi $\hat\rho$'yi düşürür. Ceza $\hat\rho=\rho$'da 0'dır, $\hat\rho\to0$ ya da 1'de sonsuza gider — L1'den farklı olarak nöronları tamamen öldürmez, hedef orana çeker.

Doğrusal AE: k = 2 kod en büyük iki özdeğerin alt uzayını tutar; hata atılan özdeğerlerin toplamı $1+0.5=1.5$ (Eckart–Young).

Kod çözücü: $(7-1)\cdot2-2\cdot1+3+1=14$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Yığılmış AE 784 → 100 → 30 → 100 → 784 (bias dahil)**

- `full`: Bağsız parametre sayısı
- `tied`: Bağlı ağırlıklarla ($W_{dec}=W_{enc}^\top$, kod çözücü bias'ları ayrı)

**Seyrek AE: hedef $\rho=0.05$, ölçülen ortalama aktivasyon $\hat\rho=0.2$**

- `kl`: $\mathrm{KL}(\rho\Vert\hat\rho)=\rho\log\frac{\rho}{\hat\rho}+(1-\rho)\log\frac{1-\rho}{1-\hat\rho}$
- `dkl`: $\partial\mathrm{KL}/\partial\hat\rho$

**Diğer**

- `pca`: Doğrusal AE, 2 boyutlu kod; veri kovaryansının özdeğerleri 5, 3, 1, 0.5 → en küçük toplam yeniden yapılandırma hatası (örnek başına beklenen)
- `up`: Konvolüsyonel kod çözücü: 7 × 7 → ConvTranspose(k = 3, s = 2, p = 1, output_padding = 1) → kenar

In [ ]:
cevaplar = dict(
    full=None,
    tied=None,
    kl=None,
    dkl=None,
    pca=None,
    up=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"full":[163814,0],"tied":[82414,0],"kl":[0.09394302602433154,0.000001],"dkl":[0.9375,1e-9],"pca":[1.5,1e-9],"up":[14,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `ae_params(sizes, tied=False)` — simetrik yığılmış AE: `sizes` kodlayıcı katman boyutları (ör. `[784, 100, 30]`), kod çözücü aynalanır.
- `kl_sparsity(rho, rho_hat)` ve `kl_sparsity_grad(rho, rho_hat)`.
- `greedy_plan(sizes)` → açgözlü katman katman eğitimin aşamaları: `[(girdi boyu, gizli boyu), …]`.
- `masking_noise(x, p, rng)` (her bileşeni p olasılıkla sıfırla) ve `gaussian_noise(x, sigma, rng)`.

In [ ]:
%%writefile stage_b.py
import math
import random


def ae_params(sizes, tied=False):
    ...


def kl_sparsity(rho, rho_hat):
    ...


def kl_sparsity_grad(rho, rho_hat):
    ...


def greedy_plan(sizes):
    ...


def masking_noise(x, p, rng):
    ...


def gaussian_noise(x, sigma, rng):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy. Bağlı ağırlıklı, sigmoid kodlu, doğrusal çıkışlı seyrek AE: $z=\sigma(XW+b)$, $\hat X=zW^\top+c$,
$$L=\frac1{2N}\|\hat X-X\|^2+\beta\sum_j\mathrm{KL}(\rho\Vert\hat\rho_j),\quad\hat\rho_j=\tfrac1N\sum_iz_{ij}.$$

- `tied_sparse_ae(X, W, b, c, rho, beta)` → `(L, dW, db, dc)` — döngüsüz; $W$ hem kodlayıcıda hem kod çözücüde: iki gradyan katkısı.
- `train(X, k, rho, beta, lr=0.5, iters=1500, seed=0)` → `(W, b, c)` — tam batch gradyan inişi (tek döngü).

In [ ]:
%%writefile stage_c.py
import numpy as np


def tied_sparse_ae(X, W, b, c, rho, beta):
    ...


def train(X, k, rho, beta, lr=0.5, iters=1500, seed=0):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Veri: sklearn rakamları (64 piksel, [0, 1]); 497 test, 1300 havuz, havuzdan 50 etiketli.

- `AE(sizes=(64, 32, 12), tied=False)` — kodlayıcı ReLU (kod katmanı doğrusal), kod çözücü aynalanmış ReLU, çıkış sigmoid; `tied=True`: kod çözücü `F.linear(h, W_enc.T, b_dec)`. `encode`, `decode`, `forward`.
- `train_ae(model, X, epochs=100, lr=3e-3, mask=0.0)` — MSE; `mask > 0`: maskeleme gürültüsü (gürültü giderici).
- `greedy_pretrain(sizes, X, epochs)` — her aşamada sığ bir AE eğit, kodla, bir sonrakine geç; sonunda ağırlıkları bir `AE(sizes)`'a kopyala.
- `classify(encoder_ae, X, y, Xt, yt, freeze, steps=300)` — `encoder_ae` None ise sıfırdan; ReLU(kod) → doğrusal 10 sınıf.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits


def digits():
    X, y = load_digits(return_X_y=True)
    return torch.tensor(X, dtype=torch.float32) / 16, torch.tensor(y)


class AE(nn.Module):
    def __init__(self, sizes=(64, 32, 12), tied=False):
        super().__init__()
        ...

    def encode(self, x):
        ...

    def decode(self, z):
        ...

    def forward(self, x):
        ...


def train_ae(model, X, epochs=100, lr=3e-3, mask=0.0):
    ...


def greedy_pretrain(sizes, X, epochs):
    ...


def classify(encoder_ae, X, y, Xt, yt, freeze, steps=300):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

16 × 16'ya büyütülmüş rakamlar.

- `ConvAE(code=16)` — Conv(1→16, 3, s2) → Conv(16→32, 3, s2) → düzleştir (32·4·4) → doğrusal → kod; tersi ConvTranspose ile.
- `DenseAE(code=16)` — 256 → 128 → kod → 128 → 256.
- Karşılaştırma: parametre, test MSE, eğitim süresi.

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvAE(nn.Module):
    def __init__(self, code=16):
        super().__init__()
        ...

    def forward(self, x):
        ...


class DenseAE(nn.Module):
    def __init__(self, code=16):
        super().__init__()
        ...

    def forward(self, x):
        ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Doğrusal gürültü giderici AE'nin kapalı form çözümü

NumPy. Merkezlenmiş veri, kovaryans Σ, doğrusal DAE $\hat x=W(x+\varepsilon)$, $\varepsilon\sim N(0,\sigma^2I)$.

- `dae_optimal(Sigma, s2)` — $W^*=\Sigma(\Sigma+\sigma^2I)^{-1}$.
- `train_linear_dae(X, s2, lr, iters, seed)` — örneklenmiş gürültüyle SGD (tam batch, her iterasyonda yeni gürültü; tek döngü).

In [ ]:
%%writefile extra_x1.py
import numpy as np


def dae_optimal(Sigma, s2):
    ...


def train_linear_dae(X, s2, lr, iters, seed):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Büzüşen (contractive) autoencoder: Jacobian cezası

NumPy. Kodlayıcı $h=\sigma(Wx+b)$, `W (k, d)`.

- `encoder_jacobian(x, W, b)` → `(k, d)`.
- `cae_penalty(X, W, b)` — örnekler üzerinden ortalama $\|J\|_F^2=\sum_j(h_j(1-h_j))^2\|W_j\|^2$; döngüsüz.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def encoder_jacobian(x, W, b):
    ...


def cae_penalty(X, W, b):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Yeniden yapılandırma hatasıyla anomali tespiti ve ROC AUC

NumPy.

- `fit_pca(X, k)` → `(ortalama, bileşenler (k, d))`; `recon_error(X, mean, comps)` → örnek başına hata.
- `roc_auc(scores, labels)` — Mann–Whitney: $\frac{\sum\text{rank}_{pos}-n_p(n_p+1)/2}{n_pn_n}$ (eşit skorlara ortalama sıra); döngüsüz.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def fit_pca(X, k):
    ...


def recon_error(X, mean, comps):
    ...


def roc_auc(scores, labels):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Vincent ve ark. (2010): yığılmış gürültü giderici AE'ler

[Vincent, Larochelle, Lajoie, Bengio, Manzagol — Stacked Denoising Autoencoders: Learning Useful Representations in a Deep Network with a Local Denoising Criterion (JMLR 2010)](https://www.jmlr.org/papers/v11/vincent10a.html)

MNIST'te 3 katmanlı SDAE'yi açgözlü katman katman eğit (her katman bir öncekinin kodunu maskeleme gürültüsüyle yeniden yapılandırır), sonra denetimli ince ayar yap. Makale Tablo 3'ün küçük sürümü: gürültü türü (maskeleme, Gauss, tuz-biber) × düzeyi için sınıflandırma hatası; ilk katman filtrelerini (Şekil 6–7) gürültü düzeyine göre görselleştir — gürültü arttıkça filtrelerin yerelleşip kenar dedektörlerine dönüştüğünü göster.


**Adımlar**

1. d aşamasındaki greedy_pretrain ve maskeleme gürültüsünü kullan.

**Kabul ölçütleri**

- Gürültüsüz AE ön eğitimine göre SDAE daha düşük test hatası.
- Filtre görselleri gürültüyle yapılanır.

## Makale ödevi: Hinton & Salakhutdinov (2006): derin AE ile boyut indirgeme

[Hinton & Salakhutdinov — Reducing the Dimensionality of Data with Neural Networks (Science 2006)](https://www.cs.toronto.edu/~hinton/absps/science.pdf)

784-1000-500-250-2 derin autoencoder'ı (RBM ön eğitimi yerine katman katman AE ön eğitimi ya da modern başlatma + Adam ile) eğit; 2B kodları PCA'nın ilk iki bileşeniyle yan yana çiz (makale Şekil 3) ve 30 boyutlu kodda yeniden yapılandırma hatasını PCA-30 ile karşılaştır (Şekil 2). Ön eğitimsiz, rastgele başlatılmış derin AE'nin (2006 koşullarında) neden takıldığını sigmoid aktivasyon + SGD ile gözle.


**Adımlar**

1. Aynı ağı ReLU + BatchNorm + Adam ile ön eğitimsiz de eğit: 2006'nın problemi bugün çözülmüş mü?

**Kabul ölçütleri**

- Derin AE 2B'de sınıfları PCA'dan belirgin daha iyi ayırır.
- 30 boyutta yeniden yapılandırma hatası PCA'dan düşük.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 17](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Hinton & Salakhutdinov — Reducing the Dimensionality of Data with Neural Networks (Science 2006)](https://www.cs.toronto.edu/~hinton/absps/science.pdf) — ©
- [Vincent ve ark. — Stacked Denoising Autoencoders (JMLR 2010)](https://www.jmlr.org/papers/v11/vincent10a.html) — ©
- [Ng — Sparse autoencoder (CS294A ders notları)](https://web.stanford.edu/class/cs294a/sparseAutoencoder.pdf) — ©
- [Rifai ve ark. — Contractive Auto-Encoders (ICML 2011)](https://icml.cc/2011/papers/455_icmlpaper.pdf) — ©
- [Bengio ve ark. — Greedy Layer-Wise Training of Deep Networks (NeurIPS 2006)](https://papers.nips.cc/paper/2006/hash/5da713a690c067105aeb2fae32403405-Abstract.html) — ©
- [UvA DL Notebooks — Tutorial 9: Autoencoders](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial9/AE_CIFAR10.html) — MIT